# <font color="#418FDE" size="6.5" uppercase>**Material Balances**</font>

>Last update: 20260609.
    
By the end of this Lecture, you will be able to:
- Represent process streams and component flow rates in Python. 
- Compute material balances for mixers, splitters, and reactors. 
- Check balance closure and identify inconsistent input data. 


## **1. Stream Models**

### **1.1. Component Flow Rates**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_04/Lecture_A/image_01_01.jpg?v=1780982849" width="250">



>* Streams contain multiple flowing components
>* Python needs named component flow values

>* Match each component to value and units
>* Reuse stream data and inspect completeness

>* Keep mass and molar flows distinct
>* Track trace components for reliable balances



In [ ]:
#@title Python Code - Component Flow Rates

# Stream dictionaries store component flow rates.
# Units must remain consistent during calculations.
# Small checks help reveal missing components.

# Define one stream using mass rates.
feed_stream = {
    "water": 120.0,
    "ethanol": 35.0,
    "salt": 5.0,
}

# Store the shared unit separately.
flow_unit = "kg/h"

# Compute the total component flow.
total_flow = sum(feed_stream.values())

# Convert component flows into mass fractions.
mass_fractions = {
    name: rate / total_flow
    for name, rate in feed_stream.items()
}

# Check whether expected components are present.
expected_components = {"water", "ethanol", "salt"}

# Compare expected names against stream keys.
missing_components = expected_components - set(feed_stream.keys())

# Print a compact stream summary.
print("Component flow rates for feed stream:")

# Print each named component rate.
for name, rate in feed_stream.items():
    fraction = mass_fractions[name]
    print(f"{name}: {rate:.1f} {flow_unit}, fraction {fraction:.3f}")

# Print total and data check.
print(f"Total flow: {total_flow:.1f} {flow_unit}")

# Report whether the stream model is complete.
if missing_components:
    print(f"Missing components: {sorted(missing_components)}")
else:
    print("All expected components are included.")



### **1.2. Stream Components**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_04/Lecture_A/image_01_02.jpg?v=1780982851" width="250">



>* Define stream components and names consistently
>* Component choices shape reliable Python balances

>* Choose species or pseudo-components wisely.
>* Match detail to calculation purpose.

>* Use consistent names for every component
>* Align streams to prevent balance errors



In [ ]:
#@title Python Code - Stream Components

# Stream components need consistent names.
# Dictionaries store component flow rates.
# Shared lists align different streams.

# Define a standard process component list.
components = ["water", "sucrose", "ethanol", "carbon_dioxide"]

# Represent streams as component-flow dictionaries.
feed_a = {"water": 90.0, "sucrose": 10.0}
feed_b = {"water": 40.0, "ethanol": 5.0}

# Convert each stream onto shared components.
def align_stream(stream, components):
    return {name: stream.get(name, 0.0) for name in components}

# Align both feeds before balance calculations.
aligned_a = align_stream(feed_a, components)
aligned_b = align_stream(feed_b, components)

# Add matching component flow rates.
mixed = {name: aligned_a[name] + aligned_b[name] for name in components}

# Check whether unknown component names appear.
unknown_names = set(feed_a) | set(feed_b)
unknown_names = unknown_names - set(components)

# Display compact stream component results.
print("Standard components:", components)
print("Feed A aligned:", aligned_a)
print("Feed B aligned:", aligned_b)
print("Mixed stream:", mixed)
print("Unknown component names:", sorted(unknown_names))



### **1.3. Molar Flow Rates**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_04/Lecture_A/image_01_03.jpg?v=1780982852" width="250">



>* Track moles crossing process boundaries over time
>* Represent each component flow consistently

>* Store each species as its own flow
>* Sum flows and compare stream compositions

>* Track units and basis for every flow
>* Consistent flows ensure reliable material balances



In [ ]:
#@title Python Code - Molar Flow Rates

# Represent stream components using molar flow rates.
# Dictionaries make component names explicit and readable.
# Totals and fractions follow from component flows.

# Define molar flows in kilomoles per hour.
stream_flows = {"N2": 78.0, "O2": 21.0, "CO2": 1.0}

# Reject negative component molar flow rates.
if any(flow < 0.0 for flow in stream_flows.values()):
    raise ValueError("Component molar flows cannot be negative.")

# Calculate the total molar flow rate.
total_flow = sum(stream_flows.values())

# Protect composition calculations from zero total flow.
if total_flow <= 0.0:
    raise ValueError("Total molar flow must be positive.")

# Convert component flows into mole fractions.
mole_fractions = {
    component: flow / total_flow
    for component, flow in stream_flows.items()
}

# Check whether mole fractions close to one.
fraction_sum = sum(mole_fractions.values())
closure_error = abs(fraction_sum - 1.0)

# Print a compact stream summary.
print("Stream basis: kmol/h")
print(f"Total molar flow: {total_flow:.2f} kmol/h")

# Print each component flow and composition.
for component, flow in stream_flows.items():
    fraction = mole_fractions[component]
    print(f"{component}: {flow:.2f} kmol/h, y = {fraction:.3f}")

# Print a simple balance closure check.
print(f"Mole fraction closure error: {closure_error:.2e}")



## **2. Unit Operation Balances**

### **2.1. Mixer Balances**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_04/Lecture_A/image_02_01.jpg?v=1780982843" width="250">



>* Mixers combine inlet streams into one outlet
>* Component and total mass are conserved

>* Add total flows and component flows separately
>* Use flow-weighted compositions, not simple averages

>* Use mixer balances to spot bad data
>* Check conservation, components, and physical intuition



In [ ]:
#@title Python Code - Mixer Balances

# Mixer balances combine incoming process streams.
# Component flows make compositions easier.
# Closure checks reveal inconsistent data.

# Define inlet streams using simple dictionaries.
stream_one = {"water": 90.0, "salt": 10.0}
stream_two = {"water": 50.0, "salt": 0.0}

# Collect all inlet streams together.
inlet_streams = [stream_one, stream_two]
components = sorted(set().union(*inlet_streams))

# Sum component flow rates entering mixer.
outlet_stream = {name: 0.0 for name in components}
for stream in inlet_streams:
    for name in components:
        outlet_stream[name] += stream.get(name, 0.0)

# Compute total outlet flow rate.
total_outlet = sum(outlet_stream.values())
total_inlet = sum(sum(stream.values()) for stream in inlet_streams)

# Convert component flows into mass fractions.
mass_fractions = {}
for name, flow in outlet_stream.items():
    mass_fractions[name] = flow / total_outlet

# Compare measured outlet with calculated outlet.
measured_outlet = {"water": 139.0, "salt": 10.0}
measured_total = sum(measured_outlet.values())
closure_error = measured_total - total_inlet

# Print compact teaching results.
print("Calculated outlet component flows, kg/h:", outlet_stream)
print("Calculated outlet total flow, kg/h:", total_outlet)
print("Outlet salt mass fraction:", round(mass_fractions["salt"], 4))
print("Measured total outlet flow, kg/h:", measured_total)
print("Closure error, kg/h:", round(closure_error, 3))
print("Balance closes within 1 kg/h:", abs(closure_error) <= 1.0)



### **2.2. Splitter Balances**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_04/Lecture_A/image_02_02.jpg?v=1780982847" width="250">



>* Splits one stream without changing species
>* Outlet compositions match; flows follow fractions

>* Simple splitters keep outlet compositions unchanged
>* Each branch carries proportional component amounts

>* Outlet flows must match inlet flows
>* Mismatches reveal errors or invalid assumptions



In [ ]:
#@title Python Code - Splitter Balances

# Splitters divide streams without changing species.
# Component flows follow the same fractions.
# Balance checks reveal inconsistent splitter data.

# Define a well mixed feed stream.
feed = {"water": 80.0, "ethanol": 20.0}

# Define outlet split fractions.
split_fractions = {"cooling": 0.30, "washing": 0.70}

# Check that fractions form a complete split.
fraction_total = sum(split_fractions.values())

# Build outlet component flow dictionaries.
outlets = {
    name: {comp: frac * flow for comp, flow in feed.items()}
    for name, frac in split_fractions.items()
}

# Compute total feed flow.
feed_total = sum(feed.values())

# Compute total outlet flow.
outlet_total = sum(sum(stream.values()) for stream in outlets.values())

# Compute each component balance residual.
residuals = {
    comp: feed[comp] - sum(stream[comp] for stream in outlets.values())
    for comp in feed
}

# Create one intentionally inconsistent measurement.
measured_washing_water = outlets["washing"]["water"] + 5.0

# Compare ideal and measured washing water.
inconsistency = measured_washing_water - outlets["washing"]["water"]

# Print concise teaching results.
print(f"Feed total: {feed_total:.1f} kg/h")
print(f"Split fraction sum: {fraction_total:.2f}")
print(f"Cooling total: {sum(outlets['cooling'].values()):.1f} kg/h")
print(f"Washing total: {sum(outlets['washing'].values()):.1f} kg/h")
print(f"Outlet total: {outlet_total:.1f} kg/h")
print(f"Water residual: {residuals['water']:.2e} kg/h")
print(f"Ethanol residual: {residuals['ethanol']:.2e} kg/h")
print(f"Measured washing water error: {inconsistency:.1f} kg/h")



### **2.3. Reactor Balances**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_04/Lecture_A/image_02_03.jpg?v=1780982845" width="250">



>* Reactions conserve total mass, not components
>* Reactants disappear while products form

>* Identify streams, components, and reactions
>* Use conversion, selectivity, yield, and stoichiometry

>* Track atoms to verify possible outlet streams.
>* Use balances to predict process needs.



In [ ]:
#@title Python Code - Reactor Balances

# Represent reactor streams with dictionaries.
# Apply stoichiometry and conversion carefully.
# Check mass closure for consistency.

# Define molecular weights in kilograms per kilomole.
mw = {
    "C2H4": 28.054,
    "H2O": 18.015,
    "C2H5OH": 46.069,
}

# Feed stream uses component molar flowrates.
feed = {
    "C2H4": 100.0,
    "H2O": 150.0,
    "C2H5OH": 0.0,
}

# Conversion consumes this ethylene fraction.
conversion = 0.80
reacted_c2h4 = feed["C2H4"] * conversion

# Stoichiometry is ethylene plus water gives ethanol.
outlet = {
    "C2H4": feed["C2H4"] - reacted_c2h4,
    "H2O": feed["H2O"] - reacted_c2h4,
    "C2H5OH": feed["C2H5OH"] + reacted_c2h4,
}

# Convert molar component flows into mass flows.
feed_mass = sum(feed[c] * mw[c] for c in feed)
outlet_mass = sum(outlet[c] * mw[c] for c in outlet)
closure_error = outlet_mass - feed_mass

# Build a compact component balance table.
print("Reactor: C2H4 + H2O -> C2H5OH")
print(f"Ethylene conversion: {conversion:.0%}")
print("Component flows, kmol/h.")

# Print each component balance clearly.
for component in feed:
    inlet_value = feed[component]
    outlet_value = outlet[component]
    print(f"{component}: in {inlet_value:.1f}, out {outlet_value:.1f}")

# Report total mass balance closure.
print(f"Feed mass: {feed_mass:.2f} kg/h")
print(f"Outlet mass: {outlet_mass:.2f} kg/h")
print(f"Closure error: {closure_error:.6f} kg/h")

# Flag inconsistent balances using tolerance.
tolerance = 1.0e-6
status = "OK" if abs(closure_error) <= tolerance else "CHECK DATA"
print(f"Mass balance status: {status}")



## **3. Balance Checks**

### **3.1. Residual Balance Checks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_04/Lecture_A/image_03_01.jpg?v=1780982854" width="250">



>* Residuals show unexplained material after balancing
>* Zero residuals confirm consistent process assumptions

>* Check both total and component residuals
>* Small total errors can hide component issues

>* Use residuals to spot data problems
>* Judge discrepancies before correcting assumptions



In [ ]:
#@title Python Code - Residual Balance Checks

# Residual checks reveal unexplained material amounts.
# Component balances can expose hidden inconsistencies.
# Small tolerances separate noise from problems.

# Define inlet component mass flow rates.
inlets = {
    "feed_A": {"water": 80.0, "ethanol": 20.0},
    "feed_B": {"water": 10.0, "ethanol": 30.0},
}

# Define reported outlet component mass flows.
outlets = {
    "product": {"water": 88.0, "ethanol": 52.0},
}

# Choose a practical closure tolerance.
tolerance = 1.0

# Collect every component mentioned anywhere.
components = sorted(
    set().union(*(stream.keys() for stream in inlets.values()),
                *(stream.keys() for stream in outlets.values()))
)

# Sum each component across inlet streams.
inlet_totals = {
    comp: sum(stream.get(comp, 0.0) for stream in inlets.values())
    for comp in components
}

# Sum each component across outlet streams.
outlet_totals = {
    comp: sum(stream.get(comp, 0.0) for stream in outlets.values())
    for comp in components
}

# Compute residuals as inlet minus outlet.
residuals = {
    comp: inlet_totals[comp] - outlet_totals[comp]
    for comp in components
}

# Compute total residual from component residuals.
total_residual = sum(residuals.values())

# Report compact residual balance results.
print("Residual balance check, kg/h")

# Print one diagnostic line per component.
for comp in components:
    status = "PASS" if abs(residuals[comp]) <= tolerance else "CHECK"
    print(f"{comp}: residual = {residuals[comp]:.1f}, {status}")

# Report overall closure using same tolerance.
total_status = "PASS" if abs(total_residual) <= tolerance else "CHECK"

# Provide one short interpretation line.
print(f"Total residual = {total_residual:.1f}, {total_status}")

# Identify the largest component discrepancy.
largest = max(residuals, key=lambda comp: abs(residuals[comp]))

# Display the likely place to investigate.
print(f"Investigate {largest} data first.")



### **3.2. Tolerance Checks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_04/Lecture_A/image_03_02.jpg?v=1780982856" width="250">



>* Set acceptable limits for small balance residuals
>* Judge imbalance by process scale and data quality

>* Use absolute and relative tolerances appropriately
>* Check totals, components, and conserved quantities

>* Set tolerances based on data and purpose
>* Investigate failures to find real inconsistencies



In [ ]:
#@title Python Code - Tolerance Checks

# Tolerance checks compare residuals against practical engineering limits.
# Small residuals can still indicate acceptable balance closure.
# Component checks reveal errors hidden by total balances.

# No additional package installation is required.

# Define absolute and relative tolerance values.
absolute_tol = 2.0
relative_tol = 0.005

# Store inlet component flow rates in kilograms per hour.
inlet = {"water": 1000.0, "ethanol": 500.0}

# Store outlet data containing one rounded inconsistent value.
outlet = {"water": 998.5, "ethanol": 497.0}

# Define a reusable closure check function.
def check_balance(name, input_flow, output_flow):
    residual = input_flow - output_flow
    scale = max(abs(input_flow), abs(output_flow), 1.0)

    limit = max(absolute_tol, relative_tol * scale)
    passed = abs(residual) <= limit

    status = "PASS" if passed else "FAIL"
    return name, residual, limit, status

# Check each component balance separately.
component_results = []
for component in inlet:
    result = check_balance(
        component,
        inlet[component],
        outlet.get(component, 0.0),
    )

    component_results.append(result)

# Check the total material balance.
total_in = sum(inlet.values())
total_out = sum(outlet.values())

total_result = check_balance("total", total_in, total_out)
all_results = component_results + [total_result]

# Print compact results for engineering interpretation.
print("Tolerance check results:")
for name, residual, limit, status in all_results:
    print(f"{name:7s}: residual={residual:5.1f}, limit={limit:4.1f}, {status}")

# Highlight why total balance alone may be misleading.
failed = [name for name, _, _, status in all_results if status == "FAIL"]

# Report the final diagnostic conclusion.
message = "Investigate: " + ", ".join(failed) if failed else "All balances pass."
print(message)



### **3.3. Missing Data**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_04/Lecture_A/image_03_03.jpg?v=1780982858" width="250">



>* Missing values can prevent balance closure
>* Data gaps do not always mean faults

>* Account for every stream and key component
>* Too many gaps make balances underdetermined

>* Check all streams before assuming process faults
>* Document data sources to guide decisions



In [ ]:
#@title Python Code - Missing Data

# Missing data can hide balance problems.
# Simple checks flag incomplete process streams.
# This example uses water evaporation data.

# Store known stream masses in kilograms.
feed_mass_kg = 1000.0
concentrate_mass_kg = 420.0
vapor_mass_kg = None

# Keep component water mass fractions together.
feed_water_frac = 0.85
concentrate_water_frac = 0.40
vapor_water_frac = 1.00

# Collect stream information in dictionaries.
inlets = {"feed": feed_mass_kg}
outlets = {"concentrate": concentrate_mass_kg, "vapor": vapor_mass_kg}

# Find streams with unavailable total mass.
missing_streams = [name for name, mass in outlets.items() if mass is None]
known_outlets = [mass for mass in outlets.values() if mass is not None]

# Check whether a direct total balance is possible.
can_check_total = len(missing_streams) == 0
total_in = sum(inlets.values())
total_known_out = sum(known_outlets)

# Estimate missing vapor from total mass balance.
estimated_vapor_kg = total_in - total_known_out
outlets["vapor"] = estimated_vapor_kg

# Compute water balance using the estimate.
water_in = feed_mass_kg * feed_water_frac
water_out = concentrate_mass_kg * concentrate_water_frac + estimated_vapor_kg
water_gap = water_in - water_out

# Decide whether estimated balance data seems consistent.
tolerance_kg = 1.0
closure_ok = abs(water_gap) <= tolerance_kg

# Print a compact missing data report.
print("Balance Checks: Missing Data")
print(f"Missing outlet streams: {missing_streams}")
print(f"Direct total balance possible: {can_check_total}")
print(f"Estimated vapor loss: {estimated_vapor_kg:.1f} kg")
print(f"Water balance gap: {water_gap:.1f} kg")
print(f"Water balance closes within tolerance: {closure_ok}")
print("Conclusion: estimate missing data before diagnosing process faults.")



# <font color="#418FDE" size="6.5" uppercase>**Material Balances**</font>


In this lecture, you learned to:
- Represent process streams and component flow rates in Python. 
- Compute material balances for mixers, splitters, and reactors. 
- Check balance closure and identify inconsistent input data. 

In the next Lecture (Lecture B), we will go over 'Energy Balances'